In [8]:
import os
import torch

print("GPU:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i))

# ── Paths ──────────────────────────────────────
BASE        = '/kaggle/input/datasets/samirhossain2001'
SAVE_DIR    = '/kaggle/working/outputs'
CACHE_DIR   = '/kaggle/working/cache'

# Training — 6 parts
TRAIN_ROOTS = [
    f"{BASE}/brats2024-trainingdatap1/training_data1_v2",
    f"{BASE}/brats2024-trainingdatap2/training_data1_v2",
    f"{BASE}/brats2024-trainingdatap3/training_data1_v2",
    f"{BASE}/brats2024-trainingdatap4/training_data1_v2",
    f"{BASE}/brats2024-trainingdatap5/training_data1_v2",
    f"{BASE}/brats2024-trainingdatap6/training_data1_v2",
]

# Test set (additional training data)
TEST_ROOT = f"{BASE}/brats2024-brats-gli-additionaltrainingdata/BraTS2024-BraTS-GLI-AdditionalTrainingData/training_data_additional"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Verify paths ──────────────────────────────
total_train = 0
for i, root in enumerate(TRAIN_ROOTS):
    count = len(os.listdir(root))
    print(f"Training part {i+1}: {count} patients")
    total_train += count

test_count = len(os.listdir(TEST_ROOT))
print(f"Test set       : {test_count} patients")
print(f"Total training : {total_train} patients")
print("✅ Paths set!")

GPU: False
GPU count: 0
Training part 1: 225 patients
Training part 2: 225 patients
Training part 3: 225 patients
Training part 4: 225 patients
Training part 5: 225 patients
Training part 6: 225 patients
Test set       : 271 patients
Total training : 1350 patients
✅ Paths set!


In [9]:
!pip install nibabel scikit-image tqdm einops -q

import numpy as np
import nibabel as nib
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
from pathlib import Path
import random
from tqdm import tqdm
import torch.nn as nn

print("✅ All imports done!")

✅ All imports done!


In [10]:
SLICE_SIZE = 128

# ── ENABLE ALL 3 VIEWS ───────────────────────
VIEWS = ["axial", "sagittal", "coronal"]

MIN_TUMOR = 0.01

CACHE_DIR = "/kaggle/working/cache"

MODALITY_SUFFIXES = {
    "t1c": "-t1c.nii",
    "t2f": "-t2f.nii",
}

SEG_SUFFIX = "-seg.nii"

# ❌ REMOVE THIS
# shutil.rmtree(CACHE_DIR, ignore_errors=True)

# ── UTILITIES ─────────────────────────────────
def load_nii(path):
    return nib.load(str(path)).get_fdata().astype(np.float32)

def normalize(vol):

    brain = vol > 0

    if brain.sum() == 0:
        return vol

    lo = np.percentile(vol[brain], 1)
    hi = np.percentile(vol[brain], 99)

    vol = np.clip(vol, lo, hi)

    vol = (vol - lo) / (hi - lo + 1e-8)

    return vol.astype(np.float32)

def binarize(seg):
    return (seg > 0).astype(np.float32)

def get_slices(vol, view):

    if view == "axial":
        return np.moveaxis(vol, 2, 0)

    elif view == "sagittal":
        return np.moveaxis(vol, 0, 0)

    elif view == "coronal":
        return np.moveaxis(vol, 1, 0)

    else:
        raise ValueError(f"Unknown view: {view}")

# ── RESIZE FUNCTIONS ──────────────────────────
def resize_img(arr, size=SLICE_SIZE):

    t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)

    return F.interpolate(
        t,
        size=(size, size),
        mode="bilinear",
        align_corners=False
    ).squeeze().numpy()

def resize_mask(arr, size=SLICE_SIZE):

    t = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)

    return F.interpolate(
        t,
        size=(size, size),
        mode="nearest"
    ).squeeze().numpy()

# ── CACHE FUNCTION ────────────────────────────
def cache_patients(patient_dirs, cache_dir, split_name):

    # Views that still need caching
    views_to_process = []

    for view in VIEWS:

        img_dir = f"{cache_dir}/{split_name}/{view}/images"
        mask_dir = f"{cache_dir}/{split_name}/{view}/masks"

        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(mask_dir, exist_ok=True)

        existing = len(os.listdir(img_dir))

        if existing > 0:

            print(
                f"✅ {split_name} {view} already cached "
                f"({existing:,} slices) — skipping"
            )

        else:
            views_to_process.append(view)

    if len(views_to_process) == 0:

        print(f"\n✅ All views already cached for {split_name}")
        return

    print(
        f"\nCaching {split_name} "
        f"views: {views_to_process} ⏳"
    )

    slice_counts = {v: 0 for v in views_to_process}
    skipped = 0

    # ── Process patients ──────────────────────
    for p in tqdm(patient_dirs, desc=split_name):

        pid = p.name

        # ── Check modality files ───────────────
        mod_paths = []
        missing = False

        for mod, suf in MODALITY_SUFFIXES.items():

            f = p / f"{pid}{suf}"

            if not f.exists():
                missing = True
                break

            mod_paths.append(f)

        if missing:
            skipped += 1
            continue

        seg_path = p / f"{pid}{SEG_SUFFIX}"

        if not seg_path.exists():
            skipped += 1
            continue

        # ── Load volumes ───────────────────────
        try:

            vols = [
                normalize(load_nii(f))
                for f in mod_paths
            ]

            seg = binarize(load_nii(seg_path))

        except Exception as e:

            print(f"\n❌ Failed loading {pid}: {e}")

            skipped += 1
            continue

        # ── Process views ──────────────────────
        for view in views_to_process:

            mod_slices = [
                get_slices(v, view)
                for v in vols
            ]

            seg_slices = get_slices(seg, view)

            for i in range(mod_slices[0].shape[0]):

                s = seg_slices[i]

                # Skip nearly-empty tumor slices
                if (s.sum() / s.size) < MIN_TUMOR:
                    continue

                # ── Image ──────────────────────
                img = np.stack([
                    resize_img(mod_slices[c][i])
                    for c in range(len(vols))
                ], axis=0)

                # ── Mask ───────────────────────
                msk = resize_mask(s).astype(np.float32)

                idx = slice_counts[view]

                # ── Save compressed ────────────
                np.savez_compressed(
                    f"{cache_dir}/{split_name}/{view}/images/{idx:06d}.npz",
                    img=img
                )

                np.savez_compressed(
                    f"{cache_dir}/{split_name}/{view}/masks/{idx:06d}.npz",
                    mask=msk
                )

                slice_counts[view] += 1

    # ── Summary ───────────────────────────────
    print(f"\n✅ {split_name} caching complete!")

    for view in views_to_process:

        total = len(os.listdir(
            f"{cache_dir}/{split_name}/{view}/images"
        ))

        print(f"  {view:10s}: {total:,} slices")

    print(f"Skipped patients: {skipped}")

# ── COLLECT TRAIN PATIENTS ───────────────────
all_train_patients = []

for root in TRAIN_ROOTS:

    all_train_patients.extend(
        sorted(Path(root).iterdir())
    )

print(f"Total training patients: {len(all_train_patients)}")

# ── COLLECT TEST PATIENTS ────────────────────
all_test_patients = sorted(
    Path(TEST_ROOT).iterdir()
)

print(f"Total test patients: {len(all_test_patients)}")

# ── CACHE TRAIN ──────────────────────────────
cache_patients(
    all_train_patients,
    CACHE_DIR,
    "train"
)

# ── OPTIONAL TEST CACHE ──────────────────────
# cache_patients(
#     all_test_patients,
#     CACHE_DIR,
#     "test"
# )

# ── VERIFY ───────────────────────────────────
print("\nFinal cache summary:")

for split in ["train"]:

    for view in VIEWS:

        img_dir = f"{CACHE_DIR}/{split}/{view}/images"

        count = len(os.listdir(img_dir))

        print(f"  {split:5s} {view:10s}: {count:,} slices")

# ── STORAGE CHECK ────────────────────────────
print("\nDisk usage:")

os.system(f"du -sh {CACHE_DIR}")

Total training patients: 1350
Total test patients: 271
✅ train axial already cached (64,777 slices) — skipping
✅ train sagittal already cached (56,813 slices) — skipping
✅ train coronal already cached (79,793 slices) — skipping

✅ All views already cached for train

Final cache summary:
  train axial     : 64,777 slices
  train sagittal  : 56,813 slices
  train coronal   : 79,793 slices

Disk usage:
11G	/kaggle/working/cache


0

In [11]:
cache_patients(
    all_test_patients,
    CACHE_DIR,
    "test"
)

✅ test axial already cached (11,593 slices) — skipping
✅ test sagittal already cached (10,734 slices) — skipping
✅ test coronal already cached (14,854 slices) — skipping

✅ All views already cached for test


In [12]:
class Augment:
    def __call__(self, img, msk):
        if random.random() > 0.5:
            img, msk = TF.hflip(img), TF.hflip(msk)
        if random.random() > 0.5:
            img, msk = TF.vflip(img), TF.vflip(msk)
        if random.random() > 0.5:
            angle = random.uniform(-15, 15)
            img = TF.rotate(img, angle, interpolation=TF.InterpolationMode.BILINEAR)
            msk = TF.rotate(msk,  angle, interpolation=TF.InterpolationMode.NEAREST)
        return img, msk

class NpyDataset(Dataset):
    def __init__(self, cache_dir, split, view, augment=False):
        self.img_dir = f"{cache_dir}/{split}/{view}/images"
        self.msk_dir = f"{cache_dir}/{split}/{view}/masks"
        self.augment = augment
        self.aug     = Augment()
        self.files   = sorted(os.listdir(self.img_dir))
        print(f"  {split:5s} {view:10s}: {len(self.files):,} slices")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]

        # ✅ FIXED ONLY THESE TWO LINES
        img = torch.from_numpy(np.load(f"{self.img_dir}/{fname}")["img"]).float()
        msk = torch.from_numpy(np.load(f"{self.msk_dir}/{fname}")["mask"]).unsqueeze(0).float()

        if self.augment:
            img, msk = self.aug(img, msk)

        return img, msk

print("✅ Dataset classes ready!")

✅ Dataset classes ready!


In [13]:
BATCH_SIZE = 8

# ── Training — all 3 views ────────────────────
print("Building training loaders...")
train_ds = ConcatDataset([
    NpyDataset(CACHE_DIR, "train", view=v, augment=True)
    for v in VIEWS
])
print(f"✅ Total training slices: {len(train_ds):,}\n")

# ── Test — all 3 views ────────────────────────
print("Building test loaders...")
test_ds = ConcatDataset([
    NpyDataset(CACHE_DIR, "test", view=v, augment=False)
    for v in VIEWS
])
print(f"✅ Total test slices: {len(test_ds):,}\n")

# ── Split 15% of training as validation ───────
val_size   = int(len(train_ds) * 0.15)
train_size = len(train_ds) - val_size
train_ds, val_ds = random_split(
    train_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
print(f"Train : {train_size:,} slices")
print(f"Val   : {val_size:,} slices")
print(f"Test  : {len(test_ds):,} slices")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

imgs, msks = next(iter(train_loader))
print(f"\n✅ Image batch: {imgs.shape}")
print(f"✅ Mask  batch: {msks.shape}")

Building training loaders...
  train axial     : 64,777 slices
  train sagittal  : 56,813 slices
  train coronal   : 79,793 slices
✅ Total training slices: 201,383

Building test loaders...
  test  axial     : 11,593 slices
  test  sagittal  : 10,734 slices
  test  coronal   : 14,854 slices
✅ Total test slices: 37,181

Train : 171,176 slices
Val   : 30,207 slices
Test  : 37,181 slices


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



✅ Image batch: torch.Size([8, 2, 128, 128])
✅ Mask  batch: torch.Size([8, 1, 128, 128])


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class EfficientMambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.norm     = nn.LayerNorm(dim)
        self.in_proj  = nn.Linear(dim, dim * 2)
        self.conv     = nn.Conv1d(dim, dim, kernel_size=3, padding=1, groups=dim)
        self.out_proj = nn.Linear(dim, dim)
        self.act      = nn.SiLU()

    def forward(self, x):
        residual = x
        x = self.norm(x)
        xz = self.in_proj(x)
        x_, z = xz.chunk(2, dim=-1)
        x_ = self.conv(x_.transpose(1,2)).transpose(1,2)
        x_ = self.act(x_)
        out = x_ * self.act(z)
        return self.out_proj(out) + residual

class VisualMambaBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.mamba = EfficientMambaBlock(dim)
        self.norm  = nn.GroupNorm(1, dim)

    def forward(self, x):
        B, C, H, W = x.shape
        x_seq = x.flatten(2).transpose(1, 2)
        x_seq = self.mamba(x_seq)
        out   = x_seq.transpose(1, 2).reshape(B, C, H, W)
        return self.norm(out) + x

class DualBranchCNNMamba(nn.Module):
    def __init__(self, in_channels=2, out_channels=1, dims=[32, 64, 128, 256]):
        super().__init__()
        self.cnn_enc1 = DoubleConv(in_channels, dims[0])
        self.cnn_enc2 = DoubleConv(dims[0], dims[1])
        self.cnn_enc3 = DoubleConv(dims[1], dims[2])
        self.cnn_enc4 = DoubleConv(dims[2], dims[3])
        self.pool     = nn.MaxPool2d(2)
        self.drop     = nn.Dropout2d(0.3)

        self.mamba_stem  = nn.Conv2d(in_channels, dims[0], 3, padding=1)
        self.mamba_enc1  = VisualMambaBlock(dims[0])
        self.mamba_enc2  = VisualMambaBlock(dims[1])
        self.mamba_enc3  = VisualMambaBlock(dims[2])
        self.mamba_enc4  = VisualMambaBlock(dims[3])
        self.mamba_down1 = nn.Conv2d(dims[0], dims[1], 2, stride=2)
        self.mamba_down2 = nn.Conv2d(dims[1], dims[2], 2, stride=2)
        self.mamba_down3 = nn.Conv2d(dims[2], dims[3], 2, stride=2)

        self.fuse1 = nn.Sequential(nn.Conv2d(dims[0]*2, dims[0], 1), nn.BatchNorm2d(dims[0]), nn.ReLU(inplace=True))
        self.fuse2 = nn.Sequential(nn.Conv2d(dims[1]*2, dims[1], 1), nn.BatchNorm2d(dims[1]), nn.ReLU(inplace=True))
        self.fuse3 = nn.Sequential(nn.Conv2d(dims[2]*2, dims[2], 1), nn.BatchNorm2d(dims[2]), nn.ReLU(inplace=True))
        self.fuse4 = nn.Sequential(nn.Conv2d(dims[3]*2, dims[3], 1), nn.BatchNorm2d(dims[3]), nn.ReLU(inplace=True))

        self.bottleneck   = DoubleConv(dims[3], dims[3]*2)
        self.mamba_bottle = VisualMambaBlock(dims[3]*2)

        self.up4   = nn.ConvTranspose2d(dims[3]*2, dims[3], 2, stride=2)
        self.dec4  = DoubleConv(dims[3]*2, dims[3])
        self.up3   = nn.ConvTranspose2d(dims[3], dims[2], 2, stride=2)
        self.dec3  = DoubleConv(dims[2]*2, dims[2])
        self.up2   = nn.ConvTranspose2d(dims[2], dims[1], 2, stride=2)
        self.dec2  = DoubleConv(dims[1]*2, dims[1])
        self.up1   = nn.ConvTranspose2d(dims[1], dims[0], 2, stride=2)
        self.dec1  = DoubleConv(dims[0]*2, dims[0])
        self.output = nn.Conv2d(dims[0], out_channels, 1)

    def forward(self, x):
        c1 = self.cnn_enc1(x)
        c2 = self.cnn_enc2(self.drop(self.pool(c1)))
        c3 = self.cnn_enc3(self.drop(self.pool(c2)))
        c4 = self.cnn_enc4(self.drop(self.pool(c3)))

        m0 = self.mamba_stem(x)
        m1 = self.mamba_enc1(m0)
        m2 = self.mamba_enc2(self.mamba_down1(m1))
        m3 = self.mamba_enc3(self.mamba_down2(m2))
        m4 = self.mamba_enc4(self.mamba_down3(m3))

        f1 = self.fuse1(torch.cat([c1, m1], dim=1))
        f2 = self.fuse2(torch.cat([c2, m2], dim=1))
        f3 = self.fuse3(torch.cat([c3, m3], dim=1))
        f4 = self.fuse4(torch.cat([c4, m4], dim=1))

        b = self.bottleneck(self.pool(f4))
        b = self.mamba_bottle(b)

        d4 = self.dec4(torch.cat([self.up4(b),  f4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), f3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), f2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), f1], dim=1))
        return self.output(d1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dual_model = DualBranchCNNMamba(in_channels=2, out_channels=1).to(device)

imgs, msks = next(iter(train_loader))
with torch.no_grad():
    preds = dual_model(imgs.to(device))

print(f"✅ Output shape: {preds.shape}")
print(f"Parameters: {sum(p.numel() for p in dual_model.parameters()):,}")
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB")

In [ ]:
import torch.optim as optim
import json
import os

# ── LOSS ─────────────────────────────────────
def dice_loss(pred, target, smooth=1):
    pred   = torch.sigmoid(pred).view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def bce_dice_loss(pred, target):
    return nn.BCEWithLogitsLoss()(pred, target) + dice_loss(pred, target)

# ── METRICS ──────────────────────────────────
def compute_metrics(pred, target, threshold=0.5, smooth=1):
    pred   = (torch.sigmoid(pred) > threshold).float().view(-1)
    target = target.view(-1)
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()
    dice      = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou       = (tp + smooth) / (tp + fp + fn + smooth)
    precision = (tp + smooth) / (tp + fp + smooth)
    recall    = (tp + smooth) / (tp + fn + smooth)
    return dice.item(), iou.item(), precision.item(), recall.item()

# ── CHECKPOINT SAVE ──────────────────────────
def save_checkpoint(model, optimizer, scheduler, epoch, best_val_dice, history):

    os.makedirs(SAVE_DIR, exist_ok=True)

    ckpt = {
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optim_state': optimizer.state_dict(),
        'sched_state': scheduler.state_dict(),
        'best_val_dice': best_val_dice,
        'history': history,
    }
    torch.save(ckpt, f"{SAVE_DIR}/checkpoint.pth")
    torch.save(ckpt, '/kaggle/working/checkpoint.pth')        # ← add this
    torch.save(model.state_dict(), '/kaggle/working/best_model.pth')  # ← add this

    print(f"  💾 Checkpoint saved → epoch {epoch} (Dice: {best_val_dice:.4f})")

# ── CHECKPOINT LOAD (FIXED) ───────────────────
def load_checkpoint(model, optimizer, scheduler):

    path = f"{SAVE_DIR}/checkpoint.pth"

    if not os.path.exists(path):
        print("🆕 No checkpoint — starting fresh")
        return 1, 0.0, {"train_loss": [], "val_loss": [], "val_dice": []}

    ckpt = torch.load(path, map_location=device)

    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optim_state'])
    scheduler.load_state_dict(ckpt['sched_state'])

    start_epoch = ckpt['epoch'] + 1
    best_val_dice = ckpt['best_val_dice']
    history = ckpt['history']

    print(f"🔄 Resumed from epoch {start_epoch} | Best Dice: {best_val_dice:.4f}")

    return start_epoch, best_val_dice, history

# ── TRAIN LOOP ───────────────────────────────
def train_model(model, train_loader, val_loader, epochs=50, lr=1e-4):

    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )

    start_epoch, best_val_dice, history = load_checkpoint(
        model, optimizer, scheduler
    )

    for epoch in range(start_epoch, epochs + 1):

        # ── TRAIN ─────────────────────────────
        model.train()
        train_loss = 0.0

        for imgs, msks in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):

            imgs, msks = imgs.to(device), msks.to(device)

            optimizer.zero_grad()

            loss = bce_dice_loss(model(imgs), msks)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # ── VALIDATION ────────────────────────
        model.eval()
        val_loss, val_dice = 0.0, 0.0

        with torch.no_grad():

            for imgs, msks in val_loader:

                imgs, msks = imgs.to(device), msks.to(device)

                preds = model(imgs)

                val_loss += bce_dice_loss(preds, msks).item()
                val_dice += compute_metrics(preds, msks)[0]

        val_loss /= len(val_loader)
        val_dice /= len(val_loader)

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_dice"].append(val_dice)

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Dice: {val_dice:.4f}"
        )

        # ── BEST MODEL ───────────────────────
        if val_dice > best_val_dice:

            best_val_dice = val_dice

            torch.save(
                model.state_dict(),
                f"{SAVE_DIR}/best_model.pth"
            )

            print(f"  ⭐ New best! (Dice: {best_val_dice:.4f})")

        # ── CHECKPOINT ───────────────────────
        save_checkpoint(
            model, optimizer, scheduler,
            epoch, best_val_dice, history
        )

    print(f"\n✅ Training complete! Best Val Dice: {best_val_dice:.4f}")

    return history, best_val_dice

print("✅ Train function ready!")

In [ ]:
print("="*55)
print("  Dual-Branch CNN-Mamba — Full BraTS 2024")
print("  Multi-View: Axial + Sagittal + Coronal")
print(f"  Training patients: 1,350")
print(f"  Test patients    : 271")
print("="*55)

history, best_dice = train_model(
    dual_model,
    train_loader,
    val_loader,
    epochs=50,
    lr=1e-4
)

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

dual_model.load_state_dict(torch.load(f"{SAVE_DIR}/best_model.pth"))
dual_model.eval()

all_dice, all_iou, all_prec, all_rec = [], [], [], []
all_preds, all_targets = [], []

with torch.no_grad():
    for imgs, msks in tqdm(test_loader, desc="Evaluating"):
        imgs, msks = imgs.to(device), msks.to(device)
        preds = dual_model(imgs)
        d, i, p, r = compute_metrics(preds, msks)
        all_dice.append(d); all_iou.append(i)
        all_prec.append(p); all_rec.append(r)
        all_preds.append(torch.sigmoid(preds).cpu().numpy().flatten())
        all_targets.append(msks.cpu().numpy().flatten())

import numpy as np
dual_dice = sum(all_dice)/len(all_dice)
dual_iou  = sum(all_iou)/len(all_iou)
dual_prec = sum(all_prec)/len(all_prec)
dual_rec  = sum(all_rec)/len(all_rec)

all_preds   = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)
fpr, tpr, _ = roc_curve(all_targets, all_preds)
dual_auc    = auc(fpr, tpr)

print("\n" + "="*45)
print("  FINAL TEST RESULTS — Full BraTS 2024")
print("="*45)
print(f"  Dice Score : {dual_dice:.4f}")
print(f"  IoU        : {dual_iou:.4f}")
print(f"  Precision  : {dual_prec:.4f}")
print(f"  Recall     : {dual_rec:.4f}")
print(f"  AUC        : {dual_auc:.4f}")
print("="*45)

In [ ]:
# ── MC Dropout Uncertainty ───────────────────
def mc_dropout_predict(model, imgs, n_passes=20):
    model.train()
    preds = []
    with torch.no_grad():
        for _ in range(n_passes):
            preds.append(torch.sigmoid(model(imgs)))
    preds       = torch.stack(preds)
    mean_pred   = preds.mean(dim=0)
    uncertainty = preds.var(dim=0)
    return mean_pred, uncertainty

imgs, msks = next(iter(test_loader))
imgs, msks = imgs.to(device), msks.to(device)
mean_pred, uncertainty = mc_dropout_predict(dual_model, imgs)
binary_pred = (mean_pred > 0.5).float()

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
col_titles = ["T1c Input", "Ground Truth", "Prediction", "Uncertainty"]
for i in range(4):
    img  = imgs[i][0].cpu().numpy()
    gt   = msks[i][0].cpu().numpy()
    pred = binary_pred[i][0].cpu().numpy()
    unc  = uncertainty[i][0].cpu().numpy()
    axes[i][0].imshow(img,  cmap="gray")
    axes[i][1].imshow(gt,   cmap="Reds", vmin=0, vmax=1)
    axes[i][2].imshow(pred, cmap="Reds", vmin=0, vmax=1)
    im = axes[i][3].imshow(unc, cmap="hot")
    plt.colorbar(im, ax=axes[i][3], fraction=0.046)
    axes[i][0].set_ylabel(f"Sample {i+1}", fontsize=9)
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontweight="bold")
for row in axes:
    for ax in row: ax.axis("off")
plt.suptitle("MC Dropout Uncertainty — Full BraTS 2024", fontsize=13)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/uncertainty_maps.png", dpi=150)
plt.show()

# ── Training Curves ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["train_loss"], label="Train Loss", color="blue")
axes[0].plot(history["val_loss"],   label="Val Loss",   color="orange")
axes[0].set_title("Loss Curve")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history["val_dice"], label="Val Dice", color="green")
axes[1].axhline(y=0.92, color="red", linestyle="--", label="Target (0.92)")
axes[1].set_title("Validation Dice Score")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_curves.png", dpi=150)
plt.show()
print("✅ All saved!")